# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Access the metadata (do not subscript, just print name and description)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and all relevant `@id`s.

The FAIR² Croissant dataset organizes data into record sets, each containing fields (columns) with unique `@id` values. We'll enumerate these entities in this section.

In [ ]:
# List all available record sets and their fields with @id

print("Available Record Sets and Fields:")
for record_set in dataset.metadata.record_sets:
    print(f"\nRecordSet @id: {record_set.id}")
    print(f"  name: {record_set.name}")
    # Display description if available
    desc = getattr(record_set, 'description', None)
    if desc:
        print(f"  description: {desc}")
    else:
        print(f"  description: (none provided)")
    print("  Fields (columns):")
    for field in record_set.fields:
        print(f"    - @id: {field.id}")
        print(f"      name: {field.name}")
        # Show the column id if present
        if hasattr(field, 'column') and field.column is not None:
            print(f"      column @id: {field.column.id if hasattr(field.column,'id') else field.column}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. We use the `@id`s of record sets found above.

Let's load all record sets into a dictionary of DataFrames for easy access. You can select a record set for further analysis by its `@id`.

In [ ]:
# Extract data from each record set using their @id
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]

dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for RecordSet {record_set_id}")

# As an example, select the first record set for demonstration
if record_sets_ids:
    example_record_set = record_sets_ids[0]
    print(f"\nDataFrame columns for RecordSet {example_record_set}:")
    print(dataframes[example_record_set].columns.tolist())
    display(dataframes[example_record_set].head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric fields, and grouping data. We'll select a numeric field using its `@id` for demonstration.

Replace the variables below with the actual `@id` values for fields you want to analyze.

In [ ]:
# Replace these example IDs with actual IDs from your record set
record_set_id = example_record_set  # From above
df = dataframes[record_set_id]

# Suppose the field @id for a numeric field is 'cr:coverage_percent' and a group field is 'cr:sample_id'
# Find suitable fields by checking printed field names and IDs above

# Let's discover numeric fields
numeric_fields = df.select_dtypes(include=['float', 'int']).columns.tolist()
if numeric_fields:
    numeric_field = numeric_fields[0]  # Use the first numeric field as example
    print(f"Using {numeric_field} as the numeric field.")

    threshold = df[numeric_field].mean()  # Use mean as threshold for illustration

    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()

    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to find a categorical/group field
    group_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if group_candidates:
        group_field = group_candidates[0]
        print(f"Grouping by {group_field}")
        grouped_df = (
            filtered_df.groupby(group_field)[numeric_field, f"{numeric_field}_normalized"].mean()
        )
        display(grouped_df.head())
    else:
        print("No suitable group field (categorical) found for grouping.")
else:
    print("No numeric fields found in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We use `matplotlib` and `seaborn` for these visualizations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the numeric field if available
if 'numeric_field' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot for grouping, if possible
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² dataset on mass spectrometry analysis of extracellular vesicles from human mast cells using `mlcroissant`.

Key steps:
- Loaded Croissant schema metadata and identified available record sets and fields (referenced by their `@id`).
- Extracted records using `mlcroissant.Dataset.records(record_set=...)` by `@id`.
- Applied exploratory data analysis, including filtering, normalization, grouping, and visualizations.

This workflow can be adapted to other Croissant-compliant datasets by updating record set and field `@id`s. For detailed investigation, refer to each field's semantics via the metadata overview.

**Happy FAIR data science!**